<a href="https://colab.research.google.com/github/Derio13/Group3A-School-Results-/blob/main/Group3A_School_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Group 3A School Results — Data Cleaning

 Week 3: Python Data Cleaning

Project: School Results Analysis  
Group: Group 3A  
Source Schema: `raw_school`  
Cleaned Schema: `group3a` Objective
Clean and prepare the school results dataset for modelling and analysis in Power BI.

### Main Data Quality Issues Identified
- Mixed date formats
- Inconsistent term labels
- Missing score values
- Invalid scores outside the 0–100 range
- Duplicate result records
- Orphan student references
- Inconsistent sex and region labels
- Missing student region values

### Tools Used
- PostgreSQL
- DBeaver
- Google Colab
- Python
- pandas
- psycopg2
- SQLAlchemy

In [1]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 51.2 MB/s eta 0:00:00


In [ ]:
import psycopg2
from getpass import getpass

host = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
port = "5432"
database = "internship"
username = "group3a"

password = getpass("9pF7Sk0MjyVPgLjoEWCI: ")

conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=username,
    password=password,
    sslmode="require"
)

print("Database connection successful!")

In [ ]:
import pandas as pd

results = pd.read_sql("SELECT * FROM raw_school.results", conn)
students = pd.read_sql("SELECT * FROM raw_school.students", conn)
subjects = pd.read_sql("SELECT * FROM raw_school.subjects", conn)
teachers = pd.read_sql("SELECT * FROM raw_school.teachers", conn)

print("results:", results.shape)
print("students:", students.shape)
print("subjects:", subjects.shape)
print("teachers:", teachers.shape)

In [ ]:
results_clean = results.copy()

results_clean.head()

In [ ]:
results_clean["term"] = (
    results_clean["term"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

results_clean["term"].value_counts()

In [ ]:
results_clean["exam_date_clean"] = pd.to_datetime(
    results_clean["exam_date"],
    errors="coerce",
    dayfirst=True
)

print(results_clean[["exam_date", "exam_date_clean"]].head(10))

print(
    "Unparsed dates:",
    results_clean["exam_date_clean"].isna().sum()
)

In [ ]:
from datetime import datetime
import pandas as pd

def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    formats = [
        "%Y-%m-%d",   # 2024-01-05
        "%Y/%m/%d",   # 2024/01/05
        "%d-%b-%Y",   # 05-Jan-2024
        "%d/%m/%Y"    # 05/01/2024
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue

    return pd.NaT

results_clean["exam_date_clean"] = results_clean["exam_date"].apply(parse_mixed_date)

print(results_clean[["exam_date", "exam_date_clean"]].head(10))
print("Unparsed dates:", results_clean["exam_date_clean"].isna().sum())

In [ ]:
results_clean.loc[
    (results_clean["score"] < 0) |
    (results_clean["score"] > 100),
    "score"
] = pd.NA

print("Missing scores after cleaning:", results_clean["score"].isna().sum())

In [ ]:
before_rows = len(results_clean)

results_clean = results_clean.drop_duplicates(
    subset=[
        "exam_date_clean",
        "student_id",
        "subject_id",
        "teacher_id",
        "term",
        "score",
        "attendance_pct"
    ],
    keep="first"
)

after_rows = len(results_clean)

print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)
print("Duplicates removed:", before_rows - after_rows)

In [ ]:
valid_student_ids = set(students["student_id"])

orphan_mask = ~results_clean["student_id"].isin(valid_student_ids)

print("Orphan student rows:", orphan_mask.sum())

results_clean.loc[
    orphan_mask,
    ["result_id", "student_id", "subject_id", "teacher_id", "term"]
].head(10)

In [ ]:

orphan_results = results_clean.loc[orphan_mask].copy()


results_clean = results_clean.loc[~orphan_mask].copy()

print("Orphan rows saved separately:", len(orphan_results))
print("Clean result rows remaining:", len(results_clean))

In [ ]:
students_clean = students.copy()

In [ ]:
students_clean["sex"] = (
    students_clean["sex"]
    .str.strip()
    .str.lower()
    .str.title()
)

students_clean["sex"].value_counts()

In [ ]:
students_clean["region"] = (
    students_clean["region"]
    .str.strip()
    .str.lower()
    .str.title()
)

students_clean["region"].value_counts(dropna=False)

In [ ]:
students_clean["enrolled_date_clean"] = students_clean["enrolled_date"].apply(parse_mixed_date)

print(
    students_clean[
        ["enrolled_date", "enrolled_date_clean"]
    ].head(10)
)

print(
    "Unparsed enrolled dates:",
    students_clean["enrolled_date_clean"].isna().sum()
)

In [ ]:
print("Students rows:", len(students_clean))
print("Missing regions:", students_clean["region"].isna().sum())
print("Sex categories:", students_clean["sex"].unique())
print("Year groups:", students_clean["year_group"].unique())
print("Unparsed enrolled dates:", students_clean["enrolled_date_clean"].isna().sum())

In [ ]:
print("SUBJECTS")
display(subjects)

print("\nMissing values in subjects:")
print(subjects.isna().sum())

print("\nDuplicate rows in subjects:")
print(subjects.duplicated().sum())


print("\nTEACHERS")
display(teachers)

print("\nMissing values in teachers:")
print(teachers.isna().sum())

print("\nDuplicate rows in teachers:")
print(teachers.duplicated().sum())

In [ ]:
teachers_clean = teachers.copy()


In [ ]:
print("Missing values in subjects:")
print(subjects.isna().sum())

print("\nDuplicate rows in subjects:")
print(subjects.duplicated().sum())

display(subjects)

In [ ]:
subjects_clean = subjects.copy()

In [ ]:
subjects_clean = subjects.copy()
teachers_clean = teachers.copy()

In [ ]:
print("===== RESULTS VALIDATION =====")
print("Rows:", len(results_clean))
print("Missing scores:", results_clean["score"].isna().sum())
print("Invalid scores:", ((results_clean["score"] < 0) | (results_clean["score"] > 100)).sum())
print("Unparsed exam dates:", results_clean["exam_date_clean"].isna().sum())
print("Term categories:", sorted(results_clean["term"].dropna().unique()))
print("Duplicate result rows:", results_clean.duplicated(
    subset=[
        "exam_date_clean",
        "student_id",
        "subject_id",
        "teacher_id",
        "term",
        "score",
        "attendance_pct"
    ]
).sum())

print("\n===== STUDENTS VALIDATION =====")
print("Rows:", len(students_clean))
print("Missing regions:", students_clean["region"].isna().sum())
print("Sex categories:", sorted(students_clean["sex"].dropna().unique()))
print("Year groups:", sorted(students_clean["year_group"].dropna().unique()))
print("Unparsed enrolled dates:", students_clean["enrolled_date_clean"].isna().sum())
print("Duplicate student rows:", students_clean.duplicated(
    subset=[
        "student_name",
        "sex",
        "year_group",
        "region",
        "enrolled_date_clean"
    ]
).sum())

print("\n===== SUBJECTS VALIDATION =====")
print("Rows:", len(subjects_clean))
print("Missing values:", subjects_clean.isna().sum().sum())
print("Duplicate rows:", subjects_clean.duplicated().sum())

print("\n===== TEACHERS VALIDATION =====")
print("Rows:", len(teachers_clean))
print("Missing values:", teachers_clean.isna().sum().sum())
print("Duplicate rows:", teachers_clean.duplicated().sum())

print("\n===== RELATIONSHIP VALIDATION =====")
print(
    "Orphan students:",
    (~results_clean["student_id"].isin(students_clean["student_id"])).sum()
)
print(
    "Orphan subjects:",
    (~results_clean["subject_id"].isin(subjects_clean["subject_id"])).sum()
)
print(
    "Orphan teachers:",
    (~results_clean["teacher_id"].isin(teachers_clean["teacher_id"])).sum()
)

In [ ]:
results_final = (
    results_clean
    .drop(columns=["exam_date"])
    .rename(columns={"exam_date_clean": "exam_date"})
    .copy()
)

students_final = (
    students_clean
    .drop(columns=["enrolled_date"])
    .rename(columns={"enrolled_date_clean": "enrolled_date"})
    .copy()
)

subjects_final = subjects_clean.copy()
teachers_final = teachers_clean.copy()

print("Results:", results_final.shape)
print("Students:", students_final.shape)
print("Subjects:", subjects_final.shape)
print("Teachers:", teachers_final.shape)

display(results_final.head())
display(students_final.head())

In [ ]:
!pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

safe_password = quote_plus(password)

engine = create_engine(
    f"postgresql+psycopg2://{username}:{safe_password}@{host}:{port}/{database}",
    connect_args={"sslmode": "require"}
)

print("SQLAlchemy engine ready.")

In [ ]:
results_final.to_sql(
    "results_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

students_final.to_sql(
    "students_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

subjects_final.to_sql(
    "subjects_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

teachers_final.to_sql(
    "teachers_clean",
    engine,
    schema="group3a",
    if_exists="replace",
    index=False
)

print("All cleaned tables written successfully to group3a.")

In [ ]:
check_query = """
SELECT
    (SELECT COUNT(*) FROM group3a.results_clean) AS results_rows,
    (SELECT COUNT(*) FROM group3a.students_clean) AS students_rows,
    (SELECT COUNT(*) FROM group3a.subjects_clean) AS subjects_rows,
    (SELECT COUNT(*) FROM group3a.teachers_clean) AS teachers_rows;
"""

pd.read_sql(check_query, engine)

# Cleaning Summary

The Week 3 cleaning process successfully prepared the school dataset for modelling.

## Results Table
- Original rows: 30,120
- Duplicate rows removed: 120
- Orphan student rows removed from the modelling dataset: 75
- Final cleaned rows: 29,925
- Invalid scores remaining: 0
- Unparsed exam dates: 0
- Duplicate result rows remaining: 0
- Orphan student, subject and teacher keys remaining: 0

## Students Table
- Final rows: 1,200
- Sex standardized to Female and Male
- Region names standardized
- Missing region values retained: 84
- Enrolled dates successfully converted
- Unparsed enrolled dates: 0
- Duplicate student rows: 0

## Subjects Table
- Final rows: 10
- Missing values: 0
- Duplicate rows: 0

## Teachers Table
- Final rows: 50
- Missing values: 0
- Duplicate rows: 0

## Database Output
The cleaned tables were written successfully to:

- `group3a.results_clean`
- `group3a.students_clean`
- `group3a.subjects_clean`
- `group3a.teachers_clean`

These tables are now ready for Week 4 data modelling in Power BI.